# JobMatch AI — NLP Resume Analysis Pipeline

This notebook explains the multi-signal matching engine behind JobMatch AI, from TF-IDF cosine similarity through skill taxonomy extraction to LLM-powered recommendations.

In [ ]:
# !pip install scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

plt.style.use('dark_background')
print("Libraries loaded ✓")

## 1. Skill Taxonomy

JobMatch uses a curated taxonomy of 200+ skills across 9 categories for structured skill extraction.

In [ ]:
SKILLS_TAXONOMY = {
    "data_analysis":    ["sql", "python", "r", "pandas", "numpy", "excel", "tableau", "power bi",
                         "data analysis", "statistical analysis", "eda"],
    "machine_learning": ["machine learning", "deep learning", "scikit-learn", "tensorflow",
                         "pytorch", "xgboost", "random forest", "neural network", "nlp",
                         "llm", "langchain", "rag", "transformers", "shap"],
    "engineering":      ["sql", "etl", "pipeline", "azure", "aws", "databricks", "spark",
                         "docker", "kubernetes", "api", "fastapi", "flask"],
    "visualization":    ["power bi", "tableau", "matplotlib", "seaborn", "plotly", "dax"],
    "soft_skills":      ["communication", "leadership", "collaboration", "agile", "scrum",
                         "project management", "stakeholder", "presentation"],
    "cloud":            ["azure", "aws", "gcp", "databricks", "snowflake", "redshift",
                         "bigquery", "s3", "lambda"],
    "databases":        ["sql", "mysql", "postgresql", "mongodb", "azure sql", "oracle",
                         "t-sql", "cte", "stored procedure"],
    "bi_tools":         ["power bi", "tableau", "looker", "qlik", "ssrs", "cognos", "dax",
                         "power query"],
    "project_mgmt":     ["jira", "agile", "scrum", "kanban", "pmp", "confluence",
                         "project management", "sprint planning"],
}
ALL_SKILLS = list({s for v in SKILLS_TAXONOMY.values() for s in v})
print(f"Total skills in taxonomy: {len(ALL_SKILLS)}")
print(f"Categories: {list(SKILLS_TAXONOMY.keys())}")

## 2. Skill Extraction

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-+#.]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def extract_skills(text):
    lower = preprocess(text)
    return list({s for s in ALL_SKILLS if re.search(r"\b" + re.escape(s) + r"\b", lower)})

# Test with a sample resume
sample_resume = """
Data Analyst with 4 years experience at Illinois DCFS.
Skills: SQL (T-SQL, CTEs, Window Functions), Python (Pandas, NumPy, Matplotlib),
Power BI (DAX, Power Query, Star Schema), Azure SQL Database, Azure Data Factory,
Databricks, JIRA, Agile/Scrum, ETL pipelines, Statistical Analysis.
Built 3 production Power BI dashboards, automated Azure SQL ETL pipelines.
M.S. Data Analytics. Strong communication and stakeholder management skills.
"""

sample_jd = """
We are looking for a Data Analyst with:
- 2+ years SQL experience (CTEs, Window Functions)
- Power BI or Tableau dashboard development
- Python for data analysis (Pandas, NumPy)
- Experience with cloud platforms (Azure or AWS)
- Strong communication and project management skills
- Experience with ETL pipelines and data warehousing
Preferred: M.S. in Data Analytics, Agile experience
"""

resume_skills = extract_skills(sample_resume)
jd_skills     = extract_skills(sample_jd)

matched = [s for s in resume_skills if s in jd_skills]
missing = [s for s in jd_skills if s not in resume_skills]
bonus   = [s for s in resume_skills if s not in jd_skills]

print(f"Resume skills ({len(resume_skills)}): {resume_skills}")
print(f"\nJD skills ({len(jd_skills)}): {jd_skills}")
print(f"\nMatched ({len(matched)}): {matched}")
print(f"Missing ({len(missing)}): {missing}")
print(f"Bonus   ({len(bonus)}):  {bonus}")
skill_score = len(matched) / len(jd_skills) if jd_skills else 0
print(f"\nSkill overlap score: {skill_score:.3f}")

## 3. TF-IDF Cosine Similarity

In [ ]:
def tfidf_similarity(text1, text2):
    vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1,2), stop_words='english')
    tfidf = vectorizer.fit_transform([preprocess(text1), preprocess(text2)])
    return float(cosine_similarity(tfidf[0], tfidf[1])[0][0])

def keyword_overlap(text1, text2):
    w1 = set(preprocess(text1).split())
    w2 = set(preprocess(text2).split())
    return len(w1 & w2) / len(w2) if w2 else 0.0

tfidf_score   = tfidf_similarity(sample_resume, sample_jd)
keyword_score = keyword_overlap(sample_resume, sample_jd)

print(f"TF-IDF cosine similarity: {tfidf_score:.4f}")
print(f"Keyword overlap ratio:    {keyword_score:.4f}")

## 4. Multi-Signal Scoring

In [ ]:
def score_match(resume, jd):
    resume_skills = extract_skills(resume)
    jd_skills     = extract_skills(jd)
    matched  = [s for s in resume_skills if s in jd_skills]
    missing  = [s for s in jd_skills if s not in resume_skills]
    bonus    = [s for s in resume_skills if s not in jd_skills]
    skill_score   = len(matched) / len(jd_skills) if jd_skills else 0.5
    tfidf_sc      = tfidf_similarity(resume, jd)
    keyword_sc    = keyword_overlap(resume, jd)
    edu_kw  = ["bachelor","master","m.s.","b.s.","phd","degree","university"]
    exp_kw  = ["years","experience","worked","developed","built","managed","led"]
    edu_sc  = sum(1 for k in edu_kw if k in resume.lower()) / len(edu_kw)
    exp_sc  = sum(1 for k in exp_kw if k in resume.lower()) / len(exp_kw)
    edu_exp = (edu_sc + exp_sc) / 2

    overall = (tfidf_sc * 0.40 + skill_score * 0.35 + keyword_sc * 0.15 + edu_exp * 0.10)
    overall = min(1.0, max(0.0, overall))

    return {
        "score": round(overall * 100),
        "skill_overlap": round(skill_score, 3),
        "tfidf_sim":     round(tfidf_sc, 3),
        "keyword_sc":    round(keyword_sc, 3),
        "edu_exp_sc":    round(edu_exp, 3),
        "matched":  matched, "missing": missing, "bonus": bonus,
        "radar": {
            "technical":  min(100, round(skill_score*100)),
            "experience": min(100, round(exp_sc*100)),
            "education":  min(100, round(edu_sc*100)),
            "keywords":   min(100, round(keyword_sc*100)),
            "culture":    min(100, round(tfidf_sc*100)),
        }
    }

result = score_match(sample_resume, sample_jd)
print(f"MATCH SCORE: {result['score']}%")
print(f"  Skill overlap:  {result['skill_overlap']} (weight: 35%)")
print(f"  TF-IDF sim:     {result['tfidf_sim']} (weight: 40%)")
print(f"  Keyword overlap:{result['keyword_sc']} (weight: 15%)")
print(f"  Edu/Exp score:  {result['edu_exp_sc']} (weight: 10%)")
print(f"\nRadar dimensions: {result['radar']}")

## 5. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Signal breakdown bar
signals = ['TF-IDF\n(40%)', 'Skill Overlap\n(35%)', 'Keywords\n(15%)', 'Edu/Exp\n(10%)']
values  = [result['tfidf_sim'], result['skill_overlap'], result['keyword_sc'], result['edu_exp_sc']]
weights = [0.40, 0.35, 0.15, 0.10]
weighted = [v*w for v,w in zip(values, weights)]

bars = axes[0].bar(signals, values, color=['#6366f1','#10b981','#f59e0b','#3b82f6'], alpha=0.85)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Score', color='#94a3b8')
axes[0].set_title(f'Signal Breakdown — Overall: {result["score"]}%', color='white')
axes[0].tick_params(colors='#94a3b8')
for spine in axes[0].spines.values(): spine.set_color('#374151')
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'{val:.2f}',
                ha='center', va='bottom', color='white', fontsize=10, fontweight='bold')

# Radar chart
dims   = list(result['radar'].keys())
vals   = [result['radar'][d]/100 for d in dims]
angles = np.linspace(0, 2*np.pi, len(dims), endpoint=False).tolist()
angles += angles[:1]; vals += vals[:1]; dims += dims[:1]

ax2 = fig.add_subplot(122, polar=True)
ax2.plot(angles, vals, color='#6366f1', linewidth=2)
ax2.fill(angles, vals, alpha=0.25, color='#6366f1')
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(dims[:-1], color='#94a3b8', size=10)
ax2.set_ylim(0, 1)
ax2.set_facecolor('#111827')
ax2.grid(color='#374151')
ax2.set_title('Match Radar', color='white', pad=15)

plt.tight_layout()
plt.savefig('../diagrams/jobmatch_pipeline.png', dpi=120, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print("Saved to diagrams/jobmatch_pipeline.png")